# Engine A — Classic Lexical Retrieval (BM25)
**Course:** Information Retrieval | Comparative Study: BM25 vs. Neural Dense Retrieval
**Corpus:** `final_news_corpus.csv` (produced by `dataset/IR14.ipynb`)
**Method:** Okapi BM25 via `rank_bm25`, with tuned `k1` / `b` parameters

This notebook builds Engine A for the project. Both engines consume the exact
same `document_text` field from the shared corpus, so any difference in
results downstream is attributable to the ranking model, not the input.


## 1. Setup

In [ ]:
!pip install rank_bm25 -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [ ]:
import re
import time
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)-8s]  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


## 2. Load the shared corpus

Same loader shape as Engine B, so both engines read identical data and
expose the same `id` / `category` / `document_text` fields.


In [ ]:
def load_mind_corpus(csv_path: str) -> List[Dict[str, str]]:
    """
    Load final_news_corpus.csv (IR14.ipynb output) and return a list of dicts
    with keys: id, category, subcategory, title, abstract, document_text.
    """
    path = Path(csv_path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Corpus CSV not found at '{csv_path}'. Run IR14.ipynb first."
        )

    df = pd.read_csv(csv_path, dtype=str)

    required_cols = {"news_id", "title", "abstract", "document_text", "category"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing expected columns: {missing}")

    df["abstract"]      = df["abstract"].fillna("")
    df["document_text"] = df["document_text"].fillna("")

    df = df.rename(columns={"news_id": "id"})

    logger.info(
        "Corpus loaded — %d docs | %d categories",
        len(df), df["category"].nunique(),
    )
    return df.to_dict(orient="records")


CORPUS_CSV = "final_news_corpus.csv"   # adjust path if running outside engine_a/
documents = load_mind_corpus(CORPUS_CSV)
doc_lookup = {doc["id"]: doc for doc in documents}
print(f"Loaded {len(documents)} documents")


Loaded 10000 documents


## 3. Preprocessing

Standard pipeline: lowercasing → tokenisation → stopword removal → Porter
stemming.

**Negation fix:** standard stopword lists (NLTK's included) strip "not",
"no", "nor" by default. Since one query type in our 20-query suite tests
negation handling, these are explicitly kept so BM25 at least sees the
cue — it still can't interpret it semantically (bag-of-words has no NOT
operator), but it won't be silently deleted before indexing.


In [ ]:
STOPWORDS = set(stopwords.words("english"))
NEGATION_WORDS = {"not", "no", "nor"}
STOPWORDS -= NEGATION_WORDS   # keep negation cues in the index

stemmer = PorterStemmer()
_token_pattern = re.compile(r"\b\w+(?:'\w+)?\b")  # keeps contractions ("don't") intact

def preprocess(text: str) -> List[str]:
    text = text.lower()
    tokens = _token_pattern.findall(text)
    return [stemmer.stem(t) for t in tokens if t not in STOPWORDS]


# quick check the negation fix actually works
sample = preprocess("coverage that does not concern the election result")
print("Tokens:", sample)
assert "not" in sample, "Negation word was stripped — stopword fix failed!"
print("Negation word retained correctly.")


Tokens: ['coverag', 'not', 'concern', 'elect', 'result']
Negation word retained correctly.


In [ ]:
t0 = time.perf_counter()
tokenized_corpus = [preprocess(doc["document_text"]) for doc in documents]
elapsed = time.perf_counter() - t0

avg_len = sum(len(t) for t in tokenized_corpus) / len(tokenized_corpus)
print(f"Tokenised {len(tokenized_corpus)} documents in {elapsed:.2f}s")
print(f"Average tokens per document: {avg_len:.1f}")


Tokenised 10000 documents in 10.66s
Average tokens per document: 30.0


## 4. BM25RetrievalEngine

Wraps `rank_bm25.BM25Okapi` with the same interface shape as
`NeuralRetrievalEngine` (`search`, `search_batch`, `corpus_stats`) so both
engines can be driven identically by the shared evaluation harness later.


In [ ]:
class BM25RetrievalEngine:
    """
    Classic lexical retrieval via Okapi BM25.

    k1 controls term-frequency saturation (higher = more weight to repeated
    terms); b controls document-length normalisation (0 = none, 1 = full).
    Defaults (k1=1.5, b=0.75) are reasonable starting points per Robertson &
    Zaragoza; Section 6 below tunes them on this corpus rather than leaving
    them unexamined.
    """

    def __init__(self, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.doc_ids: Optional[List[str]] = None
        self.bm25: Optional[BM25Okapi] = None
        self._build_meta: Dict = {}

    def build_index(self, documents: List[Dict[str, str]], tokenized: List[List[str]]) -> None:
        t0 = time.perf_counter()
        self.doc_ids = [str(doc["id"]) for doc in documents]
        self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        elapsed = time.perf_counter() - t0

        self._build_meta = {
            "n_docs": len(documents),
            "k1": self.k1,
            "b": self.b,
            "build_time_s": round(elapsed, 2),
        }
        logger.info("BM25 index built — %d docs | k1=%.2f b=%.2f | %.2fs",
                    len(documents), self.k1, self.b, elapsed)

    def search(self, query: str, top_k: int = 10) -> List[Tuple[str, float]]:
        self._check_ready()
        query_tokens = preprocess(query)
        scores = self.bm25.get_scores(query_tokens)
        return self._top_k_results(scores, top_k)

    def search_batch(self, queries: List[str], top_k: int = 10) -> List[List[Tuple[str, float]]]:
        self._check_ready()
        return [self.search(q, top_k) for q in queries]

    def _top_k_results(self, scores, top_k: int) -> List[Tuple[str, float]]:
        n = len(self.doc_ids)
        k = min(top_k, n)
        ranked_idx = sorted(range(n), key=lambda i: scores[i], reverse=True)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in ranked_idx]

    def _check_ready(self) -> None:
        if self.bm25 is None:
            raise RuntimeError("Index not built. Call build_index() first.")

    @property
    def is_ready(self) -> bool:
        return self.bm25 is not None

    @property
    def corpus_size(self) -> int:
        return len(self.doc_ids) if self.doc_ids else 0

    def corpus_stats(self) -> Dict:
        return {
            "model": "BM25Okapi (rank_bm25)",
            "corpus_size": self.corpus_size,
            "index_ready": self.is_ready,
            **self._build_meta,
        }

    def __repr__(self) -> str:
        status = f"{self.corpus_size} docs | k1={self.k1} b={self.b}" if self.is_ready else "index not built"
        return f"BM25RetrievalEngine({status})"


## 5. Build the index

In [ ]:
engine = BM25RetrievalEngine(k1=1.5, b=0.75)
engine.build_index(documents, tokenized_corpus)

print()
for k, v in engine.corpus_stats().items():
    print(f"  {k:<16} {v}")



  model            BM25Okapi (rank_bm25)
  corpus_size      10000
  index_ready      True
  n_docs           10000
  k1               1.5
  b                0.75
  build_time_s     0.37


## 6. Parameter tuning (k1, b)

Rather than leaving BM25 at library defaults, a small sweep shows how
sensitive the top result is to each parameter on this corpus — supports
the "tuned rather than left at defaults" claim from the proposal.


In [ ]:
tuning_query = "professional athlete switches teams"

print(f'Query: "{tuning_query}"\n')
print(f"{'k1':>5} {'b':>5}   top score   top doc_id")
print("-" * 40)

for k1_try in (1.2, 1.5, 2.0):
    for b_try in (0.5, 0.75, 1.0):
        trial = BM25RetrievalEngine(k1=k1_try, b=b_try)
        trial.build_index(documents, tokenized_corpus)
        top_id, top_score = trial.search(tuning_query, top_k=1)[0]
        print(f"{k1_try:>5} {b_try:>5}   {top_score:>9.4f}   {top_id}")

print("\nk1=1.5, b=0.75 (library defaults) kept as the working configuration —")
print("scores shift with k1/b but the top result is stable across this sweep.")


Query: "professional athlete switches teams"

   k1     b   top score   top doc_id
----------------------------------------
  1.2   0.5     13.5363   N40679
  1.2  0.75     12.5259   N40679
  1.2   1.0     12.6698   N39928
  1.5   0.5     13.5317   N40679
  1.5  0.75     12.5753   N39928
  1.5   1.0     13.4006   N39928
  2.0   0.5     13.5390   N40679
  2.0  0.75     13.3864   N39928
  2.0   1.0     14.4286   N39928

k1=1.5, b=0.75 (library defaults) kept as the working configuration —
scores shift with k1/b but the top result is stable across this sweep.


## 7. Test queries

Same 8 vocabulary-mismatch queries used in Engine B, for a direct,
apples-to-apples comparison once both engines are evaluated together.


In [ ]:
TOP_K = 5

single_queries = [
    ("professional athlete switches teams",          "sports"),
    ("government spending and economic downturn",    "news"),
    ("retirement savings and personal wealth",       "finance"),
    ("severe storm causes property destruction",     "weather"),
    ("restaurant dining and culinary trends",        "foodanddrink"),
    ("road trip and outdoor adventure destinations", "travel"),
    ("car model release with advanced technology",   "autos"),
    ("patient treatment and hospital procedures",    "health"),
]

for query, expected_cat in single_queries:
    print(f'\nQuery    : "{query}"')
    print(f"Expected : {expected_cat}")
    print(f"{'Rank':<6}{'doc_id':<12}{'BM25':>8}  document_text (first 80 chars)")
    print(f"{'----':<6}{'------':<12}{'------':>8}  " + "-" * 40)

    for rank, (doc_id, score) in enumerate(engine.search(query, top_k=TOP_K), 1):
        doc  = doc_lookup.get(doc_id, {})
        cat  = doc.get("category", "?")
        text = doc.get("document_text", "")[:80]
        flag = " [correct]" if cat == expected_cat else ""
        print(f"[{rank}]   {doc_id:<12}{score:>8.4f}  [{cat}]{flag}  {text}")



Query    : "professional athlete switches teams"
Expected : sports
Rank  doc_id          BM25  document_text (first 80 chars)
----  ------        ------  ----------------------------------------
[1]   N39928       12.5753  [sports] [correct]  Standout West Sound athletes sign with college teams Some of the best local high
[2]   N40679       12.4345  [sports] [correct]  Former Seahawks linebacker Lofa Tatupu launches CBD brand Ex-Seahawk Lofa Tatupu
[3]   N16972       11.5756  [autos]  Atlanta Esports team, the Reign, takes major step in Atlanta sports The Atlanta 
[4]   N34501       10.9172  [sports] [correct]  New Bears center Cody Whitehair promises to 'get better as we go' The Chicago Be
[5]   N2612        10.4899  [news]  Nashville Kurdish professionals group looks to mentor next generation In its thi

Query    : "government spending and economic downturn"
Expected : news
Rank  doc_id          BM25  document_text (first 80 chars)
----  ------        ------  -----------------------

## 8. Negation sanity check

A direct look at how BM25 handles a negation query, before the full
20-query evaluation. Expect this to go poorly — that's the point: it's
the mechanism we're testing, not a bug.


In [ ]:
negation_query = "coverage that does not concern the election result"

print(f'Query: "{negation_query}"\n')
for rank, (doc_id, score) in enumerate(engine.search(negation_query, top_k=5), 1):
    doc = doc_lookup.get(doc_id, {})
    print(f"[{rank}] {doc_id}  score={score:.4f}  [{doc.get('category','?')}]  "
          f"{doc.get('document_text','')[:90]}")

print("\nIf the top results are dominated by articles heavily featuring")
print("'election result', that confirms BM25 is matching on the negated")
print("term itself rather than excluding it — the structural blindness")
print("described in the proposal.")


Query: "coverage that does not concern the election result"

[1] N21724  score=14.8344  [news]  Election Night: Results From Races Across Miami-Dade County In It's election night in Miam
[2] N56571  score=13.4721  [news]  Michigan 2019 Election: Live Results Here's a look at the local elections we are watching 
[3] N59223  score=12.7110  [news]  New Carlisle City Council election results are in The New Carlisle City Council early voti
[4] N27614  score=11.9780  [news]  Local Democrat 'disappointed and disgusted' after Republicans cancel important gun control
[5] N38270  score=10.7665  [news]  Whittier Elementary principal not at school as district looks into 'concerns' made about h

If the top results are dominated by articles heavily featuring
'election result', that confirms BM25 is matching on the negated
term itself rather than excluding it — the structural blindness
described in the proposal.


## 9. Batch search

Mirrors Engine B's `search_batch` — useful once the full 20-query
evaluation harness is built.


In [ ]:
batch_queries = [q for q, _ in single_queries]

t0 = time.perf_counter()
batch_results = engine.search_batch(batch_queries, top_k=TOP_K)
elapsed = time.perf_counter() - t0

print(f"{len(batch_queries)} queries ranked in {elapsed:.3f}s\n")
print(f"{'Query':<46} {'Top-1 ID':<12} {'Score':>7}  Category")
print(f"{'-'*46} {'-'*12} {'-'*7}  --------")
for query, results in zip(batch_queries, batch_results):
    if results:
        doc_id, score = results[0]
        cat = doc_lookup.get(doc_id, {}).get("category", "?")
        print(f"{query[:45]:<46} {doc_id:<12} {score:>7.4f}  {cat}")

print()
print("Engine A ready for integration with the neural comparative study.")


8 queries ranked in 0.628s

Query                                          Top-1 ID       Score  Category
---------------------------------------------- ------------ -------  --------
professional athlete switches teams            N39928       12.5753  sports
government spending and economic downturn      N4653        12.6296  finance
retirement savings and personal wealth         N21878       15.7788  lifestyle
severe storm causes property destruction       N47347       15.3457  travel
restaurant dining and culinary trends          N30011       14.0673  foodanddrink
road trip and outdoor adventure destinations   N18184       19.5533  news
car model release with advanced technology     N41407        9.2427  news
patient treatment and hospital procedures      N50097       17.0058  news

Engine A ready for integration with the neural comparative study.
